# SQL → Pandas for Time-Series Forecasting

This notebook maps common **PostgreSQL** patterns used in forecasting prep to their **Pandas** equivalents.

```text
PostgreSQL result set
        ↓
Pandas DataFrame
        ↓
Datetime handling → sort → index
        ↓
Resample / groupby
        ↓
Lag & rolling features
        ↓
Calendar spine & joins
```

Focus: turn SQL-extracted sales data into a clean, time-aware dataset.


## 1. Load an SQL-style result into a DataFrame

Suppose SQL returns daily sales for one product:

| date       | product_id | sales |
| ---------- | ---------- | ----: |
| 2026-01-01 | 101        |   100 |
| 2026-01-02 | 101        |   120 |
| 2026-01-03 | 101        |   110 |

In Pandas, that becomes a `DataFrame` with the same columns.


In [1]:
import pandas as pd

df = pd.DataFrame({
    "date": ["2026-01-01", "2026-01-02", "2026-01-03"],
    "product_id": [101, 101, 101],
    "sales": [100, 120, 110],
})

df


,date,product_id,sales
0,2026-01-01,101,100
1,2026-01-02,101,120
2,2026-01-03,101,110


## 2. Inspect column types

Before time-series work, check dtypes. A date column that is still a string cannot use `.dt` accessors or `resample()`.


In [2]:
df.info()
df.dtypes


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   date        3 non-null      object
 1   product_id  3 non-null      int64 
 2   sales       3 non-null      int64 
dtypes: int64(2), object(1)
memory usage: 200.0+ bytes


date          object
product_id     int64
sales          int64
dtype: object

## 3. Convert strings to datetime

`pd.to_datetime()` is the Pandas equivalent of casting a text date to a real date/timestamp in SQL.

Until you do this, Pandas treats `date` as plain text.


In [3]:
df["date"] = pd.to_datetime(df["date"])
df.dtypes


date          datetime64[ns]
product_id             int64
sales                  int64
dtype: object

## 4. Extract calendar features with `.dt`

Once the column is `datetime64`, you can pull calendar fields:

- `df["date"].dt.year`
- `df["date"].dt.month`
- `df["date"].dt.day`
- `df["date"].dt.dayofweek`

`dayofweek` encoding:

```text
Monday    → 0
Tuesday   → 1
...
Sunday    → 6
```


In [4]:
df["day_of_week"] = df["date"].dt.dayofweek
df


,date,product_id,sales,day_of_week
0,2026-01-01,101,100,3
1,2026-01-02,101,120,4
2,2026-01-03,101,110,5


## 5. Sort by time

Lag, rolling windows, and most forecasting features assume chronological order.

Always sort before `shift()` or `rolling()`.

For multiple products, sort by `["product_id", "date"]`.


In [5]:
df = df.sort_values("date")
df


,date,product_id,sales,day_of_week
0,2026-01-01,101,100,3
1,2026-01-02,101,120,4
2,2026-01-03,101,110,5


## 6. Lag features with `shift()` — SQL `LAG()`

`shift(1)` moves values down by one row: each row gets the previous row's value.

PostgreSQL:

```sql
LAG(sales) OVER (
    ORDER BY date
)
```

Pandas:

```python
df["lag_1"] = df["sales"].shift(1)
```

The first row has no previous value → `NaN`.


In [6]:
df["lag_1"] = df["sales"].shift(1)
df


,date,product_id,sales,day_of_week,lag_1
0,2026-01-01,101,100,3,NaN
1,2026-01-02,101,120,4,100.0
2,2026-01-03,101,110,5,120.0


## 7. Set a datetime index

`resample()` works most naturally when the time column is the index.

```python
df = df.set_index("date")
```

Think of this as telling Pandas: **rows are ordered in calendar time**.


In [7]:
df = df.reset_index(drop=True)
df = df.set_index("date")
df


,product_id,sales,day_of_week,lag_1
date,,,,
2026-01-01,101,100,3,NaN
2026-01-02,101,120,4,100.0
2026-01-03,101,110,5,120.0


## 8. Change frequency with `resample()` — SQL `DATE_TRUNC`

`resample()` groups by time buckets and aggregates — similar to `DATE_TRUNC` + `GROUP BY` + `SUM`.

Common frequencies:

| Code | Meaning    |
| ---- | ---------- |
| `D`  | day        |
| `W`  | week       |
| `ME` | month-end  |

PostgreSQL monthly rollup:

```sql
SELECT
    DATE_TRUNC('month', order_date) AS month,
    SUM(quantity) AS sales
FROM orders
GROUP BY DATE_TRUNC('month', order_date);
```

Pandas:

```python
df["sales"].resample("ME").sum()
```


In [8]:
weekly_sales = df["sales"].resample("W").sum()
weekly_sales


date
2026-01-04    330
Freq: W-SUN, Name: sales, dtype: int64

In [9]:
monthly_sales = df["sales"].resample("ME").sum()
monthly_sales


date
2026-01-31    330
Freq: ME, Name: sales, dtype: int64

In [10]:
daily_sales = df["sales"].resample("D").sum()
daily_sales


date
2026-01-01    100
2026-01-02    120
2026-01-03    110
Freq: D, Name: sales, dtype: int64

## 9. Multiple products: `groupby()` + `resample()`

`groupby()` groups by **values** (like SQL `GROUP BY product_id`).  
`resample()` groups by **time**.

Together they answer: *weekly sales **per product***.

PostgreSQL:

```sql
SELECT
    DATE_TRUNC('week', order_date) AS week,
    product_id,
    SUM(quantity) AS sales
FROM orders
GROUP BY
    DATE_TRUNC('week', order_date),
    product_id;
```

Pandas:

```python
df.groupby("product_id")["sales"].resample("W").sum()
```


In [11]:
df = pd.DataFrame({
    "date": [
        "2026-01-01", "2026-01-01",
        "2026-01-02", "2026-01-02",
        "2026-01-03", "2026-01-03",
    ],
    "product_id": [101, 102, 101, 102, 101, 102],
    "sales": [100, 200, 120, 220, 110, 210],
})

df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["product_id", "date"])
df = df.set_index("date")
df


,product_id,sales
date,,
2026-01-01,101,100
2026-01-02,101,120
2026-01-03,101,110
2026-01-01,102,200
2026-01-02,102,220
2026-01-03,102,210


In [12]:
weekly_sales = (
    df
    .groupby("product_id")["sales"]
    .resample("W")
    .sum()
)

weekly_sales


product_id  date      
101         2026-01-04    330
102         2026-01-04    630
Name: sales, dtype: int64

`reset_index()` turns the MultiIndex result back into ordinary columns — often useful before joining or saving.


In [13]:
weekly_sales = weekly_sales.reset_index()
weekly_sales


,product_id,date,sales
0,101,2026-01-04,330
1,102,2026-01-04,630


## 10. Partitioned lag — `PARTITION BY` ↔ `groupby().shift()`

For one series, `shift()` is enough.  
For many products, lag must stay **within** each product — the SQL `PARTITION BY` idea.

| SQL | Pandas |
| --- | ------ |
| `PARTITION BY product_id` | `groupby("product_id")` |
| `LAG(sales)` | `.shift(1)` |

PostgreSQL:

```sql
LAG(sales, 1) OVER (
    PARTITION BY product_id
    ORDER BY date
)
```

Pandas:

```python
df.groupby("product_id")["sales"].shift(1)
```


In [14]:
df = pd.DataFrame({
    "date": ["Jan 1", "Jan 1", "Jan 2", "Jan 2"],
    "product_id": [101, 102, 101, 102],
    "sales": [100, 200, 120, 220],
})

df


,date,product_id,sales
0,Jan 1,101,100
1,Jan 1,102,200
2,Jan 2,101,120
3,Jan 2,102,220


In [15]:
df["lag_1"] = (
    df
    .groupby("product_id")["sales"]
    .shift(1)
)

df


,date,product_id,sales,lag_1
0,Jan 1,101,100,NaN
1,Jan 1,102,200,NaN
2,Jan 2,101,120,100.0
3,Jan 2,102,220,200.0


Notice product `101` on Jan 2 gets `100` (its own prior day), not product `102`'s value. That is the partition working correctly.


## 11. Rolling windows — SQL window averages

Use a **single** ordered series for rolling examples so the window does not mix products.

Default `.rolling(3).mean()` includes the **current** row (similar to `ROWS BETWEEN 2 PRECEDING AND CURRENT ROW`).


In [16]:
df = pd.DataFrame({
    "date": ["Jan 1", "Jan 2", "Jan 3", "Jan 4"],
    "sales": [100, 120, 110, 130],
})

df["rolling_mean_3"] = df["sales"].rolling(3).mean()
df


,date,sales,rolling_mean_3
0,Jan 1,100,NaN
1,Jan 2,120,NaN
2,Jan 3,110,110.0
3,Jan 4,130,120.0


To match **only past rows** (e.g. `ROWS BETWEEN 3 PRECEDING AND 1 PRECEDING`), shift first, then roll:

```python
df["sales"].shift(1).rolling(3).mean()
```

That excludes the current observation from the window.


In [17]:
df["rolling_mean_3_past"] = (
    df["sales"]
    .shift(1)
    .rolling(3)
    .mean()
)

df


,date,sales,rolling_mean_3,rolling_mean_3_past
0,Jan 1,100,NaN,NaN
1,Jan 2,120,NaN,NaN
2,Jan 3,110,110.0,NaN
3,Jan 4,130,120.0,110.0


## 12. Build a date spine — `date_range()` ≈ `generate_series()`

Forecasting often needs a complete calendar, even when some days have no sales.

PostgreSQL:

```sql
generate_series(
    DATE '2026-01-01',
    DATE '2026-01-06',
    INTERVAL '1 day'
)
```

Pandas:

```python
pd.date_range(start="2026-01-01", end="2026-01-06", freq="D")
```


In [18]:
dates = pd.date_range(
    start="2026-01-01",
    end="2026-01-06",
    freq="D",
)

dates


DatetimeIndex(['2026-01-01', '2026-01-02', '2026-01-03', '2026-01-04',
               '2026-01-05', '2026-01-06'],
              dtype='datetime64[ns]', freq='D')

## 13. Fill missing dates with `reindex()`

1. Build the full daily index with `date_range`.
2. `reindex()` to insert missing days as `NaN`.
3. Decide how to fill (`0`, forward-fill, etc.).

This is the Pandas analogue of left-joining a calendar table in SQL.


In [19]:
df = pd.DataFrame({
    "date": pd.to_datetime([
        "2026-01-01",
        "2026-01-02",
        "2026-01-03",
        "2026-01-05",  # Jan 4 missing
        "2026-01-06",
    ]),
    "sales": [100, 120, 110, 130, 140],
})

df = df.set_index("date")
df


,sales
date,
2026-01-01,100
2026-01-02,120
2026-01-03,110
2026-01-05,130
2026-01-06,140


In [20]:
full_dates = pd.date_range(
    start=df.index.min(),
    end=df.index.max(),
    freq="D",
)

df = df.reindex(full_dates)
df


,sales
2026-01-01,100.0
2026-01-02,120.0
2026-01-03,110.0
2026-01-04,NaN
2026-01-05,130.0
2026-01-06,140.0


In [21]:
df["sales"] = df["sales"].fillna(0)
df


,sales
2026-01-01,100.0
2026-01-02,120.0
2026-01-03,110.0
2026-01-04,0.0
2026-01-05,130.0
2026-01-06,140.0


## 14. Joins with `merge()` — SQL `JOIN`

`merge()` is Pandas' relational join.

PostgreSQL:

```sql
SELECT ...
FROM sales AS s
LEFT JOIN products AS p
    ON s.product_id = p.product_id;
```

| SQL join | Pandas |
| -------- | ------ |
| `INNER JOIN` | `merge(how="inner")` |
| `LEFT JOIN` | `merge(how="left")` |
| `RIGHT JOIN` | `merge(how="right")` |
| `FULL OUTER JOIN` | `merge(how="outer")` |


In [22]:
sales = pd.DataFrame({
    "date": ["Jan 1", "Jan 2"],
    "product_id": [101, 101],
    "sales": [100, 120],
})

products = pd.DataFrame({
    "product_id": [101],
    "category": ["Electronics"],
})

display(sales)
display(products)


,date,product_id,sales
0,Jan 1,101,100
1,Jan 2,101,120


,product_id,category
0,101,Electronics


In [23]:
df = sales.merge(
    products,
    on="product_id",
    how="left",
)

df


,date,product_id,sales,category
0,Jan 1,101,100,Electronics
1,Jan 2,101,120,Electronics


## Summary — SQL ↔ Pandas cheat sheet

| PostgreSQL | Pandas |
| ---------- | ------ |
| Cast / parse dates | `pd.to_datetime()` |
| `EXTRACT(DOW …)` / calendar fields | `.dt.dayofweek`, `.dt.month`, … |
| `ORDER BY date` | `sort_values("date")` |
| `DATE_TRUNC()` + aggregate | `resample()` |
| `GROUP BY` | `groupby()` |
| `LAG()` | `shift(1)` |
| `LEAD()` | `shift(-1)` |
| `PARTITION BY` + window | `groupby(...).shift()` / `.rolling()` |
| Rolling window avg | `rolling()` |
| `generate_series()` | `date_range()` |
| Left join calendar | `reindex()` |
| `LEFT JOIN` / other joins | `merge(how=...)` |

**Rule of thumb:** sort (and partition with `groupby` when needed) before any lag or rolling feature.
